# Leaf YOLO Training on Colab A100

This notebook clones the GitHub repository, mounts Google Drive, caches the dataset in Drive, prepares a one-class `leaf` dataset, trains YOLO26 candidates with an accuracy-focused fine-tune stage, evaluates on the test split, and exports ONNX/TF.js artifacts for browser deployment.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/leaf-object-detection'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_ROOT}/datasets').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_ROOT}/runs').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_ROOT}/artifacts').mkdir(parents=True, exist_ok=True)
print(f'Drive output folder: {DRIVE_ROOT}')

## 2. Clone Repo and Install Dependencies

In [ ]:
%cd /content
!rm -rf leaf-object-detection
!git clone https://github.com/fishman7337/leaf-object-detection.git
%cd /content/leaf-object-detection
!pip install -U -r requirements-colab.txt

## 3. Download or Restore Dataset Archive

This cell first checks `MyDrive/leaf-object-detection/archive.zip`. If it is missing, upload either `archive.zip` directly or your Kaggle API token `kaggle.json`. When Kaggle is used, the downloaded archive is copied into Drive for future runs.

In [ ]:
from google.colab import files
from pathlib import Path
import os
import shutil
import subprocess

drive_archive = Path(DRIVE_ROOT) / 'archive.zip'
local_archive = Path('/content/leaf-object-detection/archive.zip')

if drive_archive.exists():
    shutil.copy2(drive_archive, local_archive)
    print(f'Copied cached dataset archive from Drive: {drive_archive}')
else:
    print('Upload archive.zip or kaggle.json now.')
    uploaded = files.upload()
    if 'archive.zip' in uploaded:
        shutil.copy2('archive.zip', local_archive)
        shutil.copy2(local_archive, drive_archive)
        print(f'Uploaded archive.zip and cached it to Drive: {drive_archive}')
    elif 'kaggle.json' in uploaded:
        kaggle_dir = Path.home() / '.kaggle'
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2('kaggle.json', kaggle_dir / 'kaggle.json')
        os.chmod(kaggle_dir / 'kaggle.json', 0o600)
        subprocess.run(['pip', 'install', '-U', 'kaggle'], check=True)
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', 'sebastianpalaciob/plantvillage-for-object-detection-yolo',
            '-p', '.', '--force'
        ], check=True)
        downloaded = Path('plantvillage-for-object-detection-yolo.zip')
        shutil.move(str(downloaded), local_archive)
        shutil.copy2(local_archive, drive_archive)
        print(f'Downloaded dataset from Kaggle and cached it to Drive: {drive_archive}')
    else:
        raise FileNotFoundError('Upload either archive.zip or kaggle.json.')

print(f'Local archive ready: {local_archive} ({local_archive.stat().st_size / 1e9:.2f} GB)')

## 4. Add Mixed-Scene Public Data and Hard Negatives

In [ ]:
!mkdir -p public
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset public/PlantDoc-Object-Detection-Dataset || true
!python scripts/import_plantdoc_git.py --repo public/PlantDoc-Object-Detection-Dataset --out public/plantdoc_yolo --force
!python scripts/generate_synthetic_negatives.py --out hard_negatives/synthetic --count 300 --size 320

## 5. Prepare, Validate, and Upload Prepared Dataset to Drive

In [ ]:
!python scripts/prepare_leaf_dataset.py \
  --archive archive.zip \
  --work-dir . \
  --extra-yolo-dir public/plantdoc_yolo \
  --hard-negatives-dir hard_negatives/synthetic \
  --force

!python scripts/validate_leaf_dataset.py \
  --dataset datasets/leaf_yolo \
  --write-report

!tar -czf "$DRIVE_ROOT/datasets/leaf_yolo_dataset.tar.gz" datasets/leaf_yolo
!cp datasets/leaf_yolo/validation_report.json "$DRIVE_ROOT/datasets/validation_report.json"
!echo "Prepared dataset saved to $DRIVE_ROOT/datasets/leaf_yolo_dataset.tar.gz"

## 6. Smoke Test

In [ ]:
!python scripts/train_leaf_yolo.py \
  --data datasets/leaf_yolo/data.yaml \
  --project runs/leaf_yolo \
  --device 0 \
  --smoke \
  --export \
  --drive-out "$DRIVE_ROOT/runs/leaf_yolo"

## 7. Main Accuracy-Focused Fine-Tuning

This trains each pretrained YOLO26 candidate, then runs a lower-augmentation fine-tune stage from that model's `best.pt`. Outputs are synced to Drive after every model.

In [ ]:
!python scripts/train_leaf_yolo.py \
  --data datasets/leaf_yolo/data.yaml \
  --project runs/leaf_yolo \
  --device 0 \
  --models yolo26s.pt yolo26m.pt yolo26x.pt \
  --epochs 180 \
  --imgsz 640 \
  --fine-tune \
  --finetune-epochs 40 \
  --export \
  --drive-out "$DRIVE_ROOT/runs/leaf_yolo"

## 8. Save Final Artifacts to Drive

In [ ]:
!zip -r "$DRIVE_ROOT/artifacts/leaf_yolo_runs.zip" runs/leaf_yolo
!cp runs/leaf_yolo/training_summary.json "$DRIVE_ROOT/artifacts/training_summary.json" || true
!find "$DRIVE_ROOT/runs/leaf_yolo" -maxdepth 3 \( -name best.pt -o -name last.pt -o -name '*.onnx' \) -print
!echo "All saved under: $DRIVE_ROOT"